<a href="https://colab.research.google.com/github/GavrilLaura/PMP-2026/blob/main/Lab06/pmp_tema5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Tema 5 – Laboratorul 6**
##**Modele Markov ascunse (HMM)**

### Exercițiul 1

Notăm $o_{1:t} = (o_1, \dots, o_t)$ și $\alpha_t(j) = P(O_{1:t} = o_{1:t},\ X_t = j)$.

**a) Inițializare.** Din regula produsului:
$$\alpha_1(j) = P(O_1 = o_1, X_1 = j) = P(X_1 = j)\,P(O_1 = o_1 \mid X_1 = j) = \pi_j\, b_j(o_1).$$

**b) Recurență.** Din formula probabilității totale, condiționând după starea anterioară $X_{t-1}$:
$$\alpha_t(j) = \sum_i P(o_{1:t-1},\ X_{t-1} = i,\ X_t = j,\ O_t = o_t)$$
Aplicăm regula produsului:
$$= \sum_i P(o_{1:t-1}, X_{t-1} = i)\; P(X_t = j \mid o_{1:t-1}, X_{t-1} = i)\; P(O_t = o_t \mid X_t = j, X_{t-1} = i, o_{1:t-1})$$
Din proprietatea Markov, $P(X_t = j \mid o_{1:t-1}, X_{t-1} = i) = a_{ij}$. Din ipoteza de emisie, observația depinde doar de starea curentă: $P(O_t = o_t \mid \dots) = b_j(o_t)$. Deci:
$$\alpha_t(j) = \sum_i \alpha_{t-1}(i)\, a_{ij}\, b_j(o_t) = b_j(o_t)\sum_i \alpha_{t-1}(i)\, a_{ij}.$$

**c) Finalizare.** Din probabilitatea totală, sumând după ultima stare:
$$P(O_{1:T} = o_{1:T}) = \sum_j P(o_{1:T},\ X_T = j) = \sum_j \alpha_T(j).$$

### Exercițiul 2

In [1]:
# a) Modelul: ordinea starilor D, M, U si a notelor FB, B, S, NS
import numpy as np

stari = ["D", "M", "U"]
note = ["FB", "B", "S", "NS"]

pi = np.array([1/3, 1/3, 1/3])

A = np.array([[0.0, 0.5,  0.5 ],     # din D
              [0.5, 0.25, 0.25],     # din M
              [0.5, 0.25, 0.25]])    # din U

B = np.array([[0.10, 0.20, 0.40, 0.30],    # D
              [0.15, 0.25, 0.50, 0.10],    # M
              [0.20, 0.30, 0.40, 0.10]])   # U

print("Suma pe randuri A:", A.sum(axis=1))
print("Suma pe randuri B:", B.sum(axis=1))
print("Suma pi:", pi.sum())

# transformam notele in indici (FB -> 0, B -> 1, S -> 2, NS -> 3)
secventa = ["FB", "FB", "S", "B", "B", "S", "B", "B", "NS", "B", "B", "S"]
obs = []
for nota in secventa:
    obs.append(note.index(nota))
print("Observatii codificate:", obs)

T = len(obs)      # 12 momente
N = len(stari)    # 3 stari

Suma pe randuri A: [1. 1. 1.]
Suma pe randuri B: [1. 1. 1.]
Suma pi: 1.0
Observatii codificate: [0, 0, 2, 1, 1, 2, 1, 1, 3, 1, 1, 2]


In [2]:
# b) delta_1(j) = pi_j * b_j(o_1)
delta = np.zeros((T, N))           # scorurile
psi = np.zeros((T, N), dtype=int)  # starile precedente

for j in range(N):
    delta[0, j] = pi[j] * B[j, obs[0]]

print("delta_1 =", delta[0])

delta_1 = [0.03333333 0.05       0.06666667]


In [3]:
# c) delta_t(j) = b_j(o_t) * max_i [delta_{t-1}(i) * a_ij]
#    psi_t(j)   = argmax_i [delta_{t-1}(i) * a_ij]
for t in range(1, T):
    for j in range(N):
        cel_mai_bun_scor = -1
        cel_mai_bun_i = 0
        for i in range(N):
            scor = delta[t - 1, i] * A[i, j]
            if scor > cel_mai_bun_scor:
                cel_mai_bun_scor = scor
                cel_mai_bun_i = i
        delta[t, j] = B[j, obs[t]] * cel_mai_bun_scor
        psi[t, j] = cel_mai_bun_i

**Rolul lui ψ:** ψ_t(j) reține starea de la momentul t−1 din care s-a obținut scorul maxim δ_t(j). La final îl folosim pentru a reconstrui drumul optim mergând înapoi în timp (backtracking).

In [4]:
# d) Valorile dupa primele doua note (FB, FB)
print("delta_1 =", delta[0])
print("delta_2 =", delta[1])
print("psi_2   =", psi[1], "(0 = D, 1 = M, 2 = U)")

delta_1 = [0.03333333 0.05       0.06666667]
delta_2 = [0.00333333 0.0025     0.00333333]
psi_2   = [2 0 0] (0 = D, 1 = M, 2 = U)


In [5]:
# e) Alegem starea finala cu scor maxim, apoi mergem inapoi cu psi
drum = [0] * T

# starea finala: cea cu delta maxim la ultimul moment
for j in range(N):
    if delta[T - 1, j] > delta[T - 1, drum[T - 1]]:
        drum[T - 1] = j

# backtracking
for t in range(T - 1, 0, -1):
    drum[t - 1] = psi[t, drum[t]]

secventa_stari = []
for s in drum:
    secventa_stari.append(stari[s])

p_max = delta[T - 1, drum[T - 1]]
print("Secventa de dificultati:", secventa_stari)
print("max P(x_1..x_12, o_1..o_12) =", p_max)

Secventa de dificultati: ['D', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']
max P(x_1..x_12, o_1..o_12) = 2.1093750000000005e-11


**f)** Viterbi caută secvența completă de dificultăți cea mai probabilă, nu dificultatea cea mai probabilă separat la fiecare test. Alegerea separată a maximelor poate produce o secvență cu tranziții imposibile (de ex. D → D, care are probabilitatea 0).

Scorul de la e), ≈ 2.11 · 10⁻¹¹, este probabilitatea **comună** P(x*, o), adică a secvenței de dificultăți **și** a notelor, nu probabilitatea condiționată P(x* | o).

Observație: în acest model apar egalități de scor (ex. la δ₂(M), drumurile din D și din U au același scor), deci pot exista mai multe secvențe optime cu aceeași probabilitate.

In [6]:
# Optional: P(o) prin Forward si P(x* | o)
alfa = np.zeros((T, N))
for j in range(N):
    alfa[0, j] = pi[j] * B[j, obs[0]]

for t in range(1, T):
    for j in range(N):
        suma = 0
        for i in range(N):
            suma = suma + alfa[t - 1, i] * A[i, j]
        alfa[t, j] = B[j, obs[t]] * suma

p_obs = alfa[T - 1].sum()
print("P(o) prin Forward =", p_obs)
print("P(x* | o) = P(x*, o) / P(o) =", p_max / p_obs)

# Comparatie cu hmmlearn
!pip install -q hmmlearn
from hmmlearn import hmm

model = hmm.CategoricalHMM(n_components=3, init_params="")
model.startprob_ = pi
model.transmat_ = A
model.emissionprob_ = B

obs_coloana = np.array(obs).reshape(-1, 1)
log_p, drum_lib = model.decode(obs_coloana, algorithm="viterbi")

drum_lib_nume = []
for s in drum_lib:
    drum_lib_nume.append(stari[s])
print("hmmlearn:", drum_lib_nume, " P =", np.exp(log_p))
print("P(o) hmmlearn =", np.exp(model.score(obs_coloana)))

P(o) prin Forward = 7.325942897172459e-08
P(x* | o) = P(x*, o) / P(o) = 0.0002879322197302604
hmmlearn: ['U', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']  P = 2.109375000000024e-11
P(o) hmmlearn = 7.32594289717248e-08
